## Step 1:- Imports
- pandas
- os
- HuggingFaceEmbeddings from langchain_community.embeddings
- Chroma from langchain_community.vectorstores
- Document from langchain_core.documents.

In [9]:
import pandas as pd
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_core.documents import Document
import os

## Step 2 :- Load your chunked dataset csv in a dataframe

In [10]:
df=pd.read_csv("../../data/chunked/langChainchunks.csv")

## Step 3 :- Create an empty list to store all of the documents in the csv

In [12]:
docs=[]

## Step 4 :- Using a for loop for iterating over all of the ``rows``
- define a for loop which fetches index, row (use df.iterrows())
- inside the loop, we create a variable which uses Document which we imported in ``Step 1``.
- page_content for embeddings
- metadata for everything else except chunks since we are creating embeddings for the chunks.

> Use the following for metadata easy copy paste:-

metadata={
            "chunk_id"        : str(row["chunk_id"]),
            "source"          : str(row["source"]),
            "title"           : str(row["title"]),
            "url"             : str(row["url"]),
            "published"       : str(row["published"])
        }

- After everything has been iterated over, append the variable we created inside for loop to the empty list created in ``Step 3``.

In [13]:
for i, j in df.iterrows():
    doc=Document(page_content=j["chunk_text"],
                metadata={
            "chunk_id"        : str(j["chunk_id"]),
            "source"          : str(j["source"]),
            "title"           : str(j["title"]),
            "url"             : str(j["url"]),
            "published"       : str(j["published"])
        } )
    docs.append(doc)

## Step 5:- 
- Create a variable which stores the settings for the model such as:- 
    - model_name
    - model_kwargs={"device":"cpu"}
    - encode_kwargs={"normalize_embeddings":"True"}

In [14]:
model=HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device":"cpu"},
    encode_kwargs={"normalize_embeddings":"True"}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5475.77it/s]


## Step 7:- 
- Create a chroma save directory.
    - Please use mine:- ``"../../data/chromaDB/chromadbLangChain"``
- Then do a os.mkdirs(path, exist_ok=True)

In [15]:
save_dir="../../data/chromaDB/chromadbLangChain"
os.makedirs(save_dir, exist_ok=True)

## Step 8 :- Using from_documents :- 
- create a variable.
- In that variable call in Chroma.from_document()
- Define the parameters such that:-
    - documents= Empty List we created in `step 3`,
    - embedding = variable name we created in `step 5`,
    - persist_directory = variable we created in `step 7`,
    - collection_name = any name.

### ``NOTES:- ``
- Chroma.from_documents() does three things in one call:
   1. Calls embedding_model.embed_documents() on every page_content
   2. Creates a ChromaDB collection internally
   3. Stores vectors + metadata + original text together

- persist_directory tells LangChain where to save the ChromaDB files.
- collection_name gives the collection a name inside ChromaDB.

- Key difference from FAISS:
  - FAISS needed a separate .save_local() call after from_documents().
  - Chroma saves to disk automatically during from_documents()
  - because persist_directory is passed in at creation time.

In [16]:
vectors=Chroma.from_documents(
    documents=docs,
    embedding=model,
    persist_directory=save_dir,
    collection_name="vikrant_chroma_langchain"
)